# 02. Wyniki zadań Spark: analiza statystyczna i modele MLlib

Notatnik wizualizuje pliki zapisane przez zadania Spark:

| Plik | Zadanie Spark |
|---|---|
| `reports/analysis.json` | `src/mushroom/analysis.py` (χ², FP-Growth) |
| `reports/metrics.json` | `src/mushroom/train.py` (porównanie i strojenie modeli) |

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REP = ROOT / "reports"
FIG = REP / "figures"
load = lambda n: json.loads((REP / n).read_text(encoding="utf-8"))
analysis, metrics = load("analysis.json"), load("metrics.json")

def save(name):
    plt.savefig(FIG / f"{name}.png", bbox_inches="tight")

## 1. Analiza statystyczna w Spark

In [ ]:
chi = pd.DataFrame(analysis["chi_square"])
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=chi, y="feature", x="cramers_v", color="#2f6b4f", ax=ax)
ax.set_title("Siła związku cechy z klasą (V Craméra, Spark ChiSquareTest)")
save("06_cramers_v")
plt.show()
chi

### Reguły asocjacyjne FP-Growth

Każdy rekord to koszyk elementów `cecha=wartość` oraz `class=p|e`. Pokazujemy reguły, których następnikiem jest klasa (pomijamy reguły, które tylko dokładają warunki do reguły już pokazanej).

In [ ]:
for cls, name in [("p", "trujący"), ("e", "jadalny")]:
    t = pd.DataFrame(analysis["association_rules"][cls])
    t["antecedent"] = t["antecedent"].apply(" ∧ ".join)
    print(f"Reguły → {name}")
    display(t)

## 2. Porównanie modeli MLlib

In [ ]:
rnd = pd.DataFrame(metrics["random"]).set_index("model")
sp = pd.DataFrame(metrics["species_holdout"]).set_index("model")
cmp = pd.DataFrame({
    "acc_losowy": rnd.accuracy, "auc_losowy": rnd.auc,
    "acc_gatunki": sp.accuracy, "auc_gatunki": sp.auc,
    "czułość_gatunki": sp.recall_poisonous, "trujące→jadalne_gatunki": sp.poisonous_as_edible,
})
print(metrics["data"])
cmp

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(cmp))
ax.bar(x - 0.2, cmp.acc_losowy, 0.4, label="losowy podział wierszy", color="#9b9282")
ax.bar(x + 0.2, cmp.acc_gatunki, 0.4, label="gatunki niewidziane w treningu", color="#2f6b4f")
ax.axhline(0.556, color="#b03a2e", ls=":", label="zawsze \"trujący\" (test gatunków)")
ax.set_xticks(x, cmp.index); ax.set_ylim(0.4, 1.01); ax.set_ylabel("accuracy")
ax.legend(loc="lower left"); ax.set_title("Accuracy: losowy podział vs. gatunki niewidziane")
save("07_model_comparison")
plt.show()

### Strojenie lasu losowego (CrossValidator z foldami po gatunkach)

Każdy gatunek trafia w całości do jednego z 4 foldów, więc walidacja mierzy jakość dla gatunków nieznanych.

In [ ]:
pd.DataFrame(metrics["tuning"])

Wszystkie punkty siatki mają CV AUC 0,66-0,68, więc zmiana hiperparametrów prawie nic nie daje na nowych gatunkach. Najlepsze ustawienie (100 drzew, głębokość 12) to to samo, którego używamy w porównaniu i w strumieniu.

## 3. Próg bezpieczeństwa (teza H3)

Lepiej uznać jadalny grzyb za trujący niż odwrotnie, więc szukamy najwyższego progu, przy którym na znanych gatunkach model nie przepuszcza żadnego trującego grzyba.

In [ ]:
thr = metrics["model"]["threshold"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (key, title) in zip(axes, [("known", "gatunki znane (losowy podział)"), ("unseen", "gatunki niewidziane")]):
    c = pd.DataFrame(metrics["threshold_curve"][key])
    ax.plot(c.threshold, c.recall_poisonous, marker="o", color="#b03a2e", label="czułość dla trujących")
    ax.plot(c.threshold, c.specificity, marker="o", color="#2f6b4f", label="swoistość (jadalne rozpoznane)")
    ax.axvline(thr, color="black", ls=":", label=f"próg w strumieniu = {thr}")
    ax.set_xlabel("próg P(trujący)"); ax.set_ylim(0, 1.02); ax.set_title(title)
axes[0].legend(loc="lower left")
plt.tight_layout()
save("08_threshold")
plt.show()
pd.concat({k: pd.DataFrame(v).set_index("threshold") for k, v in metrics["threshold_curve"].items()}, axis=1)